# Perché l'algebra lineare è fondamentale nell'analisi dei dati?

**Metodi numerici per l'analisi dei dati**  
Secondo anno — Informatica per il Management

---

Quando sentiamo parlare di algebra lineare pensiamo spesso a vettori, matrici e sistemi di equazioni. Nell'analisi dei dati, però, questi oggetti non sono soltanto concetti astratti:

> **sono il linguaggio con cui rappresentiamo i dati e costruiamo gli algoritmi che li analizzano.**

L'obiettivo di questo notebook non è spiegare tutta l'algebra lineare, ma mostrare **perché vale la pena studiarla**.

## Partiamo da un problema concreto

Un'azienda raccoglie alcune informazioni sui propri clienti:

- numero di visite al sito nell'ultimo mese;
- tempo medio trascorso sul sito;
- spesa mensile;
- numero di acquisti.

Ogni cliente è descritto da più numeri. Possiamo organizzare questi dati in una tabella.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

clienti = pd.DataFrame({
    "visite": [5, 12, 8, 20, 16, 4],
    "tempo_medio_min": [2.1, 5.4, 3.2, 8.1, 6.7, 1.8],
    "spesa_euro": [25, 110, 55, 210, 160, 18],
    "acquisti": [1, 4, 2, 7, 5, 1]
}, index=["Cliente A", "Cliente B", "Cliente C", "Cliente D", "Cliente E", "Cliente F"])

clienti

## 1. Un dataset è una matrice

La tabella precedente può essere vista come una matrice $X$:

$$
X = \begin{bmatrix}
5 & 2.1 & 25 & 1\\
12 & 5.4 & 110 & 4\\
\vdots & \vdots & \vdots & \vdots
\end{bmatrix}.
$$

- ogni **riga** rappresenta un cliente;
- ogni **colonna** rappresenta una caratteristica;
- ogni cliente è quindi anche un **vettore** di numeri.

Questa semplice osservazione è già una prima risposta:

> Per analizzare un dataset dobbiamo saper lavorare con vettori e matrici.

La stessa rappresentazione si usa per dati economici, questionari, sensori, testi, immagini e moltissimi altri tipi di informazione.

In [ ]:
X = clienti.to_numpy()

print("Dimensione della matrice:", X.shape)
print("Numero di clienti:", X.shape[0])
print("Numero di caratteristiche per cliente:", X.shape[1])

## 2. L'algebra lineare permette di confrontare gli oggetti

Molte domande dell'analisi dei dati hanno una natura geometrica:

- quali clienti hanno comportamenti simili?
- quali prodotti vengono acquistati insieme?
- quali documenti trattano argomenti vicini?
- quali osservazioni sono anomale?

Se ogni cliente è un vettore, la **distanza tra vettori** misura quanto due clienti sono diversi. Il prodotto scalare e l'angolo tra vettori permettono invece di misurare la loro similarità.

Nel grafico seguente consideriamo solo due caratteristiche per poter visualizzare i clienti come punti nel piano.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(clienti["visite"], clienti["spesa_euro"], s=90, color="#2563eb")

for nome, riga in clienti.iterrows():
    ax.annotate(nome[-1], (riga["visite"], riga["spesa_euro"]),
                xytext=(6, 5), textcoords="offset points", fontsize=11)

ax.set_xlabel("Visite al sito")
ax.set_ylabel("Spesa mensile (€)")
ax.set_title("I clienti diventano punti: la similarità diventa geometria")
plt.show()

Nel grafico si vede, per esempio, che i clienti **D** ed **E** sono relativamente vicini, mentre **D** ed **F** sono molto lontani.

L'algebra lineare trasforma quindi un'idea qualitativa — *questi clienti si assomigliano* — in una quantità che un algoritmo può calcolare.

Questa idea è alla base di:

- segmentazione della clientela;
- sistemi di raccomandazione;
- ricerca di contenuti simili;
- algoritmi di clustering e classificazione.

## 3. I modelli diventano operazioni tra matrici e vettori

Supponiamo di voler prevedere la spesa di un cliente usando alcune sue caratteristiche. Un modello lineare assegna un peso a ogni caratteristica:

$$
\text{spesa prevista}
= w_0 + w_1\,\text{visite} + w_2\,\text{tempo} + w_3\,\text{acquisti}.
$$

Per un solo cliente è una somma pesata. Per tutti i clienti contemporaneamente diventa un prodotto matrice-vettore:

$$
\widehat{\boldsymbol y}=X\boldsymbol w.
$$

Con una sola operazione possiamo quindi produrre migliaia o milioni di previsioni.

In [ ]:
# Un semplice modello dimostrativo: i coefficienti sono fissati solo per mostrare
# come un prodotto matrice-vettore produca tutte le previsioni insieme.
caratteristiche = clienti[["visite", "tempo_medio_min", "acquisti"]].to_numpy()
X_modello = np.column_stack([np.ones(len(clienti)), caratteristiche])
pesi = np.array([2.0, 1.2, 3.0, 22.0])

spesa_prevista = X_modello @ pesi

pd.DataFrame({
    "spesa_reale": clienti["spesa_euro"],
    "spesa_prevista": np.round(spesa_prevista, 1)
})

Il simbolo `@` indica il prodotto tra una matrice e un vettore. Non è importante, per ora, sapere come scegliere i pesi migliori: ciò che conta è osservare che **il modello è espresso nel linguaggio dell'algebra lineare**.

La regressione lineare conduce poi a un problema di **minimi quadrati**. Anche modelli molto più complessi, comprese le reti neurali, eseguono continuamente prodotti tra matrici e vettori.

## 4. Ridurre la complessità significa trovare nuove direzioni

I dataset reali possono avere centinaia o migliaia di caratteristiche. Alcune sono ridondanti, altre contengono poco segnale.

La **riduzione dimensionale** cerca poche nuove variabili che conservino la parte più importante dell'informazione. La PCA, per esempio, individua direzioni lungo le quali i dati variano maggiormente e proietta i dati su queste direzioni.

Geometricamente, significa sostituire molte coordinate con poche coordinate più informative.

In [ ]:
rng = np.random.default_rng(7)
x = np.linspace(-3, 3, 80)
y = 1.5 * x + rng.normal(0, 0.7, size=x.size)
dati = np.column_stack([x, y])

# Prima direzione principale, calcolata qui solo per visualizzarne il significato.
dati_centrati = dati - dati.mean(axis=0)
_, _, Vt = np.linalg.svd(dati_centrati, full_matrices=False)
direzione = Vt[0]
t = np.array([-4, 4])
retta = dati.mean(axis=0) + np.outer(t, direzione)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(dati[:, 0], dati[:, 1], alpha=0.65, color="#0f766e")
ax.plot(retta[:, 0], retta[:, 1], color="#dc2626", linewidth=3,
        label="direzione che riassume meglio i dati")
ax.set_xlabel("caratteristica 1")
ax.set_ylabel("caratteristica 2")
ax.set_title("Ridurre la dimensione: cercare le direzioni importanti")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.show()

I punti del grafico hanno due coordinate, ma sono distribuiti soprattutto lungo una direzione. Proiettandoli sulla linea rossa possiamo descriverli quasi completamente con **un solo numero**.

Autovalori, autovettori e decomposizione ai valori singolari (SVD) non sono quindi calcoli fini a sé stessi: aiutano a

- visualizzare dati con molte variabili;
- comprimere l'informazione;
- eliminare rumore e ridondanza;
- rendere più veloci gli algoritmi successivi.

## 5. Anche immagini e testi diventano vettori e matrici

Un'immagine in scala di grigi è una matrice: ogni elemento contiene l'intensità di un pixel. Un'immagine a colori può essere rappresentata mediante tre matrici, una per ciascun canale RGB.

Per questo motivo, elaborare un'immagine significa spesso eseguire operazioni su matrici.

In [ ]:
# Una piccola immagine artificiale 12 x 12.
immagine = np.zeros((12, 12))
immagine[2:10, 3:9] = 0.35
immagine[4:8, 5:7] = 1.0
immagine[5:7, 4:8] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(immagine, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Immagine")
axes[0].axis("off")

axes[1].imshow(immagine, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("La stessa immagine come matrice di numeri")
for i in range(immagine.shape[0]):
    for j in range(immagine.shape[1]):
        axes[1].text(j, i, f"{immagine[i, j]:.1f}", ha="center", va="center", fontsize=6)
axes[1].set_xticks([])
axes[1].set_yticks([])
plt.tight_layout()
plt.show()

Anche oggetti non immediatamente numerici vengono trasformati in vettori:

- un testo può essere rappresentato da un vettore di frequenze o da un *embedding*;
- un utente da un vettore di preferenze;
- un prodotto da un vettore di caratteristiche;
- una serie temporale da una sequenza di vettori.

Una volta ottenuta questa rappresentazione, possiamo applicare gli stessi strumenti matematici a dati molto diversi.

## 6. Perché gli strumenti devono anche essere numericamente affidabili?

In teoria possiamo scrivere una formula corretta; in pratica il computer lavora con precisione finita e dataset molto grandi. Alcuni problemi amplificano piccoli errori o piccole perturbazioni nei dati.

Studiare l'algebra lineare in un corso di **metodi numerici** serve anche a capire:

- quale algoritmo usare;
- quanto costa in tempo e memoria;
- se il risultato è sensibile agli errori nei dati;
- quando una soluzione calcolata è davvero affidabile.

Non basta quindi sapere che una soluzione esiste: bisogna saperla calcolare in modo efficiente e stabile.

## Una mappa delle idee

| Domanda sui dati | Idea di algebra lineare |
|---|---|
| Come rappresento molte osservazioni e caratteristiche? | Vettori e matrici |
| Quali oggetti si assomigliano? | Distanze, norme, prodotto scalare |
| Come trasformo simultaneamente tutti i dati? | Prodotto matrice-vettore |
| Come costruisco una previsione? | Sistemi lineari e minimi quadrati |
| Ci sono caratteristiche ridondanti? | Rango e dipendenza lineare |
| Come riassumo dati ad alta dimensione? | Autovettori, SVD e PCA |
| Come elaboro immagini, testi e segnali? | Operazioni tra vettori e matrici |
| Posso fidarmi del risultato numerico? | Condizionamento e stabilità |

## Messaggio conclusivo

L'algebra lineare è importante nell'analisi dei dati perché consente di:

1. **rappresentare** i dati in una forma utilizzabile dal computer;
2. **interpretare geometricamente** relazioni, distanze e similarità;
3. **costruire modelli** e calcolare molte previsioni contemporaneamente;
4. **ridurre e comprimere** grandi quantità di informazione;
5. **progettare algoritmi** efficienti e numericamente affidabili.

> Studiare l'algebra lineare non significa soltanto imparare a fare calcoli con le matrici. Significa imparare il linguaggio che permette di trasformare i dati in informazione.

Nelle lezioni successive vedremo come ciascuna di queste idee si traduce in strumenti e algoritmi concreti.